# PyTorch: Redes Neuronales, Datasets y Checkpoints

Este cuadernillo consolida los conceptos más importantes de tres notebooks:

| Origen | Tema principal |
|--------|----------------|
| Notebook 02 | Diseño de redes: Sequential, Module, residual, capas, optimizadores |
| Notebook 03 | Datos: tensor directo → Dataset → DataLoader → collate_fn → checkpoints |
| Notebook 04 | Exportación: state_dict, modelo completo, TorchScript, ONNX |

---

In [44]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from tqdm import tqdm
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

#Se utilizo el dataset de Kaggle "Challenges in Representation Learning: Facial Expression Recognition Challenge" que se encuentra en el archivo 'train.csv'
#https://www.kaggle.com/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/data

Dispositivo: cpu


---
# PARTE 1 — Redes Neuronales en PyTorch
*(Conceptos del Notebook 02)*

## 1.1 Modelo Secuencial (`torch.nn.Sequential`)

La forma más sencilla de definir una red: una secuencia de capas donde la salida de una es la entrada de la siguiente. Ideal para un **MLP** (Perceptrón Multicapa).

In [46]:
D_in, H, D_out = 784, 100, 10

# Modelo MLP secuencial: entrada -> capa oculta (ReLU) -> salida
modelo_seq = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

# Los modelos PyTorch siempre esperan [batch_size, features] como entrada
salida = modelo_seq(torch.randn(64, 784))
print(f'Forma de salida: {salida.shape}')  # [64, 10]

Forma de salida: torch.Size([64, 10])


## 1.2 Modelo Personalizado (`torch.nn.Module`)

Cuando necesitamos control total sobre el flujo de datos (conexiones residuales, múltiples entradas/salidas, etc.), creamos una clase que hereda de `torch.nn.Module`.

**Dos métodos obligatorios:**
- `__init__`: define las capas
- `forward`: define la lógica de cálculo

In [49]:
class MLP(torch.nn.Module):
    """MLP equivalente al Sequential anterior, pero con control total."""
    def __init__(self, D_in, H, D_out):
        super(MLP, self).__init__()
        # Definimos las capas en __init__
        self.fc1  = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2  = torch.nn.Linear(H, D_out)

    def forward(self, x):
        # Definimos el flujo de datos en forward
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

modelo_mlp = MLP(784, 100, 10)
print(modelo_mlp)

MLP(
  (fc1): Linear(in_features=784, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=10, bias=True)
)


## 1.3 Modelo con Conexiones Residuales

Las conexiones residuales **no se pueden expresar con Sequential**.
La idea: sumar la entrada de un bloque con su salida antes de pasar al siguiente.
Esto mejora el flujo de gradientes y es la base de arquitecturas como ResNet.

In [51]:
class MLPResidual(torch.nn.Module):
    """MLP con conexión residual: suma entrada y salida de la capa oculta."""
    def __init__(self, D_in, H, D_out):
        super(MLPResidual, self).__init__()
        self.fc1  = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2  = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)          # guardamos la salida de fc1
        x  = self.relu(x1)
        x  = self.fc2(x + x1)     # sumamos x1 (conexión residual)
        return x

modelo_res = MLPResidual(784, 100, 10)
salida = modelo_res(torch.randn(16, 784))
print(f'Salida con residual: {salida.shape}')

Salida con residual: torch.Size([16, 10])


## 1.4 Acceso y Modificación de Capas

PyTorch permite acceder a capas por nombre, leer sus pesos y reemplazarlas.
Esto es la base del **Transfer Learning**: reutilizar capas entrenadas y cambiar solo la capa final.

In [53]:
# Acceder a una capa por nombre
print('Capa fc1:', modelo_res.fc1)

# Acceder a los tensores de pesos y bias
print('Pesos fc1:', modelo_res.fc1.weight.shape)
print('Bias fc1 :', modelo_res.fc1.bias.shape)

# Reemplazar una capa (transfer learning: cambiar la salida a 1 clase)
modelo_res.fc2 = torch.nn.Linear(100, 1)
print('\nModelo después de reemplazar fc2:')
print(modelo_res)

# Restauramos fc2 original para los siguientes ejemplos
modelo_res.fc2 = torch.nn.Linear(100, 10)

# Obtener lista de capas (útil para construir subredes)
capas = list(modelo_res.children())
print(f'\nCapas del modelo: {capas}')

# Crear nueva red excluyendo la última capa (extractor de características)
extractor = torch.nn.Sequential(*list(modelo_res.children())[:-1])
print(f'Extractor (sin fc2):\n{extractor}')

Capa fc1: Linear(in_features=784, out_features=100, bias=True)
Pesos fc1: torch.Size([100, 784])
Bias fc1 : torch.Size([100])

Modelo después de reemplazar fc2:
MLPResidual(
  (fc1): Linear(in_features=784, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

Capas del modelo: [Linear(in_features=784, out_features=100, bias=True), ReLU(), Linear(in_features=100, out_features=10, bias=True)]
Extractor (sin fc2):
Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): ReLU()
)


## 1.5 Actualización Manual de Pesos vs Optimizador

Es fundamental entender **qué hace un optimizador internamente** antes de usarlo.
Primero lo implementamos a mano, luego usamos `torch.optim` que automatiza esto.

In [20]:
# Funciones de pérdida implementadas manualmente (para entender la teoría)

def softmax(x):
    """Convierte logits en distribución de probabilidad."""
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

def cross_entropy(output, target):
    """Pérdida de entropía cruzada: mide distancia entre predicción y etiqueta real."""
    logits = output[torch.arange(len(output)), target]
    loss   = -logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    return loss.mean()

In [21]:
# Preparamos datos MNIST como tensores (forma más básica)
from sklearn.datasets import fetch_openml

mnist   = fetch_openml('mnist_784', version=1)
X, Y    = mnist['data'], mnist['target']
x_arr   = np.array(X)
y_arr   = np.array(Y)

X_train, X_test = x_arr[:60000] / 255., x_arr[60000:] / 255.
y_train, y_test = y_arr[:60000].astype(np.int32), y_arr[60000:].astype(np.int32)

# Tensores en GPU
X_t = torch.from_numpy(X_train).float().to(device)
Y_t = torch.from_numpy(y_train).long().to(device)

print(f'X_t: {X_t.shape}, Y_t: {Y_t.shape}')

X_t: torch.Size([60000, 784]), Y_t: torch.Size([60000])


In [22]:
# ── ENTRENAMIENTO CON ACTUALIZACIÓN MANUAL DE PESOS ──────────────────────────
# Esto ilustra exactamente lo que hace un optimizador internamente.

modelo = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

lr     = 0.8
epochs = 10
l      = []
modelo.train()

for e in range(1, epochs + 1):
    # 1. Forward
    y_pred = modelo(X_t)

    # 2. Pérdida manual
    loss = cross_entropy(y_pred, Y_t)
    l.append(loss.item())

    # 3. Poner gradientes a cero (evita acumulación entre épocas)
    modelo.zero_grad()

    # 4. Backpropagation: PyTorch calcula todos los gradientes automáticamente
    loss.backward()

    # 5. Actualización manual: param = param - lr * grad
    with torch.no_grad():  # desactivamos autograd para no registrar esta operación
        for param in modelo.parameters():
            param -= lr * param.grad

    if not e % 2:
        print(f'Época {e}/{epochs}  Loss: {np.mean(l):.5f}')

Época 2/10  Loss: 2.24315
Época 4/10  Loss: 2.09307
Época 6/10  Loss: 1.89404
Época 8/10  Loss: 1.77092
Época 10/10  Loss: 1.73058


In [23]:
# ── ENTRENAMIENTO CON torch.optim Y torch.nn (forma recomendada) ──────────────
# torch.nn.CrossEntropyLoss y torch.optim.SGD automatizan lo anterior.

modelo = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

criterion = torch.nn.CrossEntropyLoss()          # función de pérdida
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.8)  # optimizador

epochs = 10
l      = []
modelo.train()

for e in range(1, epochs + 1):
    y_pred = modelo(X_t)               # forward
    loss   = criterion(y_pred, Y_t)    # pérdida
    l.append(loss.item())

    optimizer.zero_grad()              # poner gradientes a cero
    loss.backward()                    # backprop
    optimizer.step()                   # actualizar pesos

    if not e % 2:
        print(f'Época {e}/{epochs}  Loss: {np.mean(l):.5f}')

Época 2/10  Loss: 2.25300
Época 4/10  Loss: 2.09797
Época 6/10  Loss: 1.88871
Época 8/10  Loss: 1.71083
Época 10/10  Loss: 1.77144


In [24]:
from sklearn.metrics import accuracy_score

def evaluar_mlp(model, X_test_np, y_test_np):
    """Evaluación para modelos que reciben tensores planos."""
    model.eval()   # desactiva Dropout/BatchNorm en inferencia
    with torch.no_grad():
        x_t    = torch.from_numpy(X_test_np).float().to(device)
        y_pred = model(x_t)
        y_prob = softmax(y_pred)
        clases = torch.argmax(y_prob, axis=1).cpu().numpy()
    return accuracy_score(y_test_np, clases)

acc = evaluar_mlp(modelo, X_test, y_test)
print(f'Precisión MLP: {acc:.4f}')

Precisión MLP: 0.4312


---
# PARTE 2 — Datasets y DataLoaders
*(Conceptos del Notebook 03)*

## 2.1 Evolución: tensor directo → mini-batches manuales → Dataset → DataLoader

### Nivel 1: Tensor directo (batch gradient descent)
Todo el dataset en memoria, un solo paso por época. No escala.

In [26]:
# Nivel 1: Todo el dataset en cada paso (ya visto arriba con X_t, Y_t)
# Desventaja: si el dataset no cabe en GPU, falla.
# Desventaja: el gradiente usa toda la info, converge lento.
print('Nivel 1: Tensor directo')
print(f'  X_t en GPU: {X_t.shape} — todos los datos en cada step')

Nivel 1: Tensor directo
  X_t en GPU: torch.Size([60000, 784]) — todos los datos en cada step


### Nivel 2: Mini-batches manuales
Dividimos el dataset en chunks y actualizamos pesos con cada chunk.

In [27]:
modelo2  = torch.nn.Sequential(
    torch.nn.Linear(D_in, H), torch.nn.ReLU(), torch.nn.Linear(H, D_out)
).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(modelo2.parameters(), lr=0.8)

batch_size = 100
batches    = len(X_t) // batch_size
epochs     = 5
l          = []
modelo2.train()

for e in range(1, epochs + 1):
    _l = []
    for b in range(batches):
        # Slicing manual del tensor
        x_b = X_t[b * batch_size:(b + 1) * batch_size]
        y_b = Y_t[b * batch_size:(b + 1) * batch_size]

        y_pred = modelo2(x_b)
        loss   = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    print(f'Época {e}/{epochs}  Loss: {np.mean(l):.5f}')

# Desventaja: código frágil, no hace shuffle, requiere reescribir para cada dataset

Época 1/5  Loss: 0.35347
Época 2/5  Loss: 0.24945
Época 3/5  Loss: 0.20280
Época 4/5  Loss: 0.17423
Época 5/5  Loss: 0.15418


### Nivel 3: Clase `Dataset`

Encapsulamos la lógica de acceso a datos en una clase con **tres métodos obligatorios**:
- `__init__`: carga o referencia los datos
- `__len__`: número total de muestras
- `__getitem__(ix)`: devuelve la muestra `ix`

In [28]:
class MNISTDataset(torch.utils.data.Dataset):
    """
    Dataset personalizado para MNIST.
    Convierte arrays NumPy a tensores en GPU durante la construcción.
    """
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float().to(device)
        self.Y = torch.from_numpy(Y).long().to(device)

    def __len__(self):
        return len(self.X)  # número total de muestras

    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]  # muestra individual o slice

dataset_train = MNISTDataset(X_train, y_train)
dataset_test  = MNISTDataset(X_test,  y_test)

print(f'Tamaño dataset train : {len(dataset_train)}')
print(f'Forma de una muestra : {dataset_train[0][0].shape}')  # (784,)
print(f'Etiqueta de muestra 0: {dataset_train[0][1]}')

Tamaño dataset train : 60000
Forma de una muestra : torch.Size([784])
Etiqueta de muestra 0: 5


### Nivel 4: `DataLoader`

El `DataLoader` recibe un `Dataset` y gestiona automáticamente:
- División en mini-batches (`batch_size`)
- Mezcla aleatoria al inicio de cada época (`shuffle=True`)
- Carga paralela con múltiples workers (`num_workers`)

El bucle de entrenamiento se simplifica a `for x_b, y_b in dataloader`.

In [29]:
dataloader_train = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=100,
    shuffle=True    # mezcla al inicio de cada época
)
dataloader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=100,
    shuffle=False
)

# Verificar forma de un batch
x_b, y_b = next(iter(dataloader_train))
print(f'Batch X: {x_b.shape}')  # [100, 784]
print(f'Batch Y: {y_b.shape}')  # [100]

Batch X: torch.Size([100, 784])
Batch Y: torch.Size([100])


In [30]:
# Entrenamiento usando DataLoader (forma limpia y recomendada)
modelo3   = torch.nn.Sequential(
    torch.nn.Linear(D_in, H), torch.nn.ReLU(), torch.nn.Linear(H, D_out)
).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(modelo3.parameters(), lr=0.8)

epochs = 5
l      = []
modelo3.train()

for e in range(1, epochs + 1):
    _l = []
    for x_b, y_b in dataloader_train:   # ← DataLoader gestiona los batches
        y_pred = modelo3(x_b)
        loss   = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    print(f'Época {e}/{epochs}  Loss: {np.mean(l):.5f}')

Época 1/5  Loss: 0.30752
Época 2/5  Loss: 0.21441
Época 3/5  Loss: 0.17219
Época 4/5  Loss: 0.14701
Época 5/5  Loss: 0.12954


## 2.2 `collate_fn`: lógica personalizada para armar batches

Por defecto, el DataLoader apila muestras con `torch.stack`. Podemos sobreescribir
esto para aplicar transformaciones, padding de secuencias, data augmentation, etc.

In [47]:
def collate_fn(batch):
    """
    Función personalizada para ensamblar un batch.
    `batch` es una lista de tuplas (x, y) retornadas por __getitem__.
    Aquí simplemente apilamos — pero podríamos aplicar augmentation, padding, etc.
    """
    xs = torch.stack([x for x, y in batch])
    ys = torch.stack([y for x, y in batch])
    return xs, ys

dataloader_custom = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=100,
    shuffle=True,
    collate_fn=collate_fn  # ← función personalizada
)

x_b, y_b = next(iter(dataloader_custom))
print(f'Batch con collate_fn — X: {x_b.shape}, Y: {y_b.shape}')

Batch con collate_fn — X: torch.Size([100, 784]), Y: torch.Size([100])


---
# PARTE 3 — Modelo CNN, Entrenamiento Completo y Checkpoints
*(Conceptos del Notebook 04)*

## 3.1 Dataset con torchvision y transformaciones

Cuando usamos imágenes, `torchvision` provee datasets y transformaciones listas para usar.
Las transformaciones se aplican on-the-fly en cada batch (no se guarda el dataset transformado).

In [48]:
import pandas as pd
from PIL import Image

class FER2013Dataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, transform=None): # Quitamos el parámetro 'split'
        df = pd.read_csv(csv_file)
        # Como todo el CSV es de entrenamiento (o prueba), guardamos todo
        self.data = df 
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        label = int(self.data.iloc[idx]['emotion'])
        pixels = self.data.iloc[idx]['pixels']
        # Usamos np.fromstring que es más rápido y seguro para estos casos
        image_np = np.fromstring(pixels, sep=' ', dtype=np.uint8).reshape(48, 48)
        image = Image.fromarray(image_np)
        
        if self.transform:
            image = self.transform(image)
        return image, label

transform_fer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Normalización genérica
])

# Se asume que train.csv está en la misma carpeta
dataloaders = {
    'train': torch.utils.data.DataLoader(
        FER2013Dataset('train.csv', transform=transform_fer), # Archivo de train
        batch_size=128, shuffle=True, pin_memory=True
    ),
    'test': torch.utils.data.DataLoader(
        FER2013Dataset('train.csv', transform=transform_fer),  # Archivo de test
        batch_size=128, shuffle=False, pin_memory=True
    )
}

x_b, y_b = next(iter(dataloaders['train']))
print(f'Batch imágenes FER: {x_b.shape}') # Debería ser [128, 1, 48, 48]

Batch imágenes FER: torch.Size([128, 1, 48, 48])


c:\Users\X13\Desktop\1-2026\Inteligencia Artificial I\Laboratorio\.env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 3.2 CNN (Red Neuronal Convolucional)

Para imágenes usamos capas convolucionales que preservan la estructura espacial.
- `Conv2d`: aplica filtros que detectan patrones locales
- `MaxPool2d`: reduce dimensiones espaciales
- `view` / `flatten`: aplana para la capa final fully connected

In [38]:
def bloque_conv(c_in, c_out, k=3, p=1, s=1, pk=2, ps=2):
    return torch.nn.Sequential(
        torch.nn.Conv2d(c_in, c_out, k, padding=p, stride=s),
        torch.nn.BatchNorm2d(c_out), # Añadido para estabilizar
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(pk, stride=ps)
    )

class CNN(torch.nn.Module):
    def __init__(self, n_canales=1, n_clases=7): # 7 emociones
        super().__init__()
        self.conv1 = bloque_conv(n_canales, 64)   
        self.conv2 = bloque_conv(64, 128)         
        self.conv3 = bloque_conv(128, 256)        
        self.fc    = torch.nn.Linear(256 * 6 * 6, n_clases)

    def forward(self, x):
        x = self.conv1(x) 
        x = self.conv2(x) 
        x = self.conv3(x) 
        x = x.view(x.shape[0], -1) 
        return self.fc(x)

cnn_prueba = CNN()
with torch.no_grad():
    out = cnn_prueba(torch.randn(8, 1, 48, 48)) # Imagen de 48x48
print(f'Salida CNN: {out.shape}')  # Debería ser [8, 7]

Salida CNN: torch.Size([8, 7])


## 3.3 Función `fit` con checkpoints del mejor modelo

Incorporamos el patrón clave del Notebook 04:
- Guardar `state_dict` **solo cuando mejora** la precisión en validación
- Al terminar, **cargar automáticamente el mejor checkpoint**
- Guardar también el estado del optimizador para poder reanudar el entrenamiento

In [39]:
def fit(model, dataloaders, epochs=5, lr=1e-3, checkpoint_path='./mejor_modelo.pt'):
    """
    Entrena el modelo guardando el mejor checkpoint según val_acc.

    Patrón del Notebook 04:
    - Guarda state_dict cuando val_acc mejora
    - Al terminar, carga el mejor modelo automáticamente
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()
    mejor_acc = 0.0

    for epoch in range(1, epochs + 1):
        # ── Entrenamiento ──────────────────────────────────
        model.train()
        train_loss, train_acc = [], []
        bar = tqdm(dataloaders['train'])

        for X, y in bar:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            y_hat = model(X)
            loss  = criterion(y_hat, y)
            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())
            acc = (y == torch.argmax(y_hat, axis=1)).float().mean().item()
            train_acc.append(acc)
            bar.set_description(f'loss {np.mean(train_loss):.5f}  acc {np.mean(train_acc):.5f}')

        # ── Validación ─────────────────────────────────────
        model.eval()
        val_loss, val_acc = [], []
        bar = tqdm(dataloaders['test'])
        with torch.no_grad():
            for X, y in bar:
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                loss  = criterion(y_hat, y)
                val_loss.append(loss.item())
                acc = (y == torch.argmax(y_hat, axis=1)).float().mean().item()
                val_acc.append(acc)
                bar.set_description(f'val_loss {np.mean(val_loss):.5f}  val_acc {np.mean(val_acc):.5f}')

        val_acc_mean = np.mean(val_acc)

        # ── Checkpoint: guardar solo si es el mejor ────────
        if val_acc_mean > mejor_acc:
            mejor_acc = val_acc_mean
            torch.save(model.state_dict(), checkpoint_path)
            print(f'  ✓ Mejor modelo guardado (época {epoch}, val_acc={val_acc_mean:.5f})')

        print(f'Época {epoch}/{epochs}  '
              f'loss {np.mean(train_loss):.5f}  '
              f'val_loss {np.mean(val_loss):.5f}  '
              f'acc {np.mean(train_acc):.5f}  '
              f'val_acc {val_acc_mean:.5f}')

    # ── Cargar el mejor modelo al final ───────────────────
    model.load_state_dict(torch.load(checkpoint_path))
    print(f'\nEntrenamiento completo. Mejor val_acc: {mejor_acc:.5f}')

In [40]:
CHECKPOINT = './mejor_cnn.pt'

modelo_cnn = CNN()
fit(modelo_cnn, dataloaders, epochs=5, lr=1e-3, checkpoint_path=CHECKPOINT)

val_loss 1.56806  val_acc 0.40182: 100%|██████████| 225/225 [01:26<00:00,  2.60it/s]


  ✓ Mejor modelo guardado (época 1, val_acc=0.40182)
Época 1/5  loss 1.92025  val_loss 1.56806  acc 0.33147  val_acc 0.40182


val_loss 1.31803  val_acc 0.50026: 100%|██████████| 225/225 [01:23<00:00,  2.71it/s]


  ✓ Mejor modelo guardado (época 2, val_acc=0.50026)
Época 2/5  loss 1.46630  val_loss 1.31803  acc 0.45843  val_acc 0.50026


val_loss 1.18500  val_acc 0.56625: 100%|██████████| 225/225 [01:26<00:00,  2.60it/s]


  ✓ Mejor modelo guardado (época 3, val_acc=0.56625)
Época 3/5  loss 1.27151  val_loss 1.18500  acc 0.52506  val_acc 0.56625


val_loss 1.20811  val_acc 0.55588: 100%|██████████| 225/225 [01:26<00:00,  2.61it/s]


Época 4/5  loss 1.17420  val_loss 1.20811  acc 0.56179  val_acc 0.55588


val_loss 0.98229  val_acc 0.63695: 100%|██████████| 225/225 [01:26<00:00,  2.60it/s]


  ✓ Mejor modelo guardado (época 5, val_acc=0.63695)
Época 5/5  loss 1.07633  val_loss 0.98229  acc 0.60273  val_acc 0.63695

Entrenamiento completo. Mejor val_acc: 0.63695


## 3.4 Evaluación final

In [41]:
def evaluate(model, dataloader):
    """Evalúa el modelo. Usa model.eval() y torch.no_grad() para inferencia."""
    model.eval()
    model.to(device)
    bar  = tqdm(dataloader)
    acc  = []
    with torch.no_grad():
        for X, y in bar:
            X, y  = X.to(device), y.to(device)
            y_hat = model(X)
            acc.append((y == torch.argmax(y_hat, axis=1)).float().mean().item())
            bar.set_description(f'acc {np.mean(acc):.5f}')
    print(f'Precisión final: {np.mean(acc):.5f}')

evaluate(modelo_cnn, dataloaders['test'])

acc 0.63695: 100%|██████████| 225/225 [01:30<00:00,  2.48it/s]

Precisión final: 0.63695


---
# PARTE 4 — Guardado y Exportación de Modelos
*(Conceptos del Notebook 04)*

## 4.1 Opciones de guardado: `state_dict` vs modelo completo

PyTorch ofrece dos formas de guardar un modelo. La diferencia es importante:

In [45]:
# ── Opción A: Guardar solo state_dict (RECOMENDADA) ──────────────────────────
# Guarda únicamente los pesos. Requiere tener la clase del modelo disponible al cargar.
# Más ligero, más flexible, permite guardar info extra.

torch.save(modelo_cnn.state_dict(), './pesos_cnn.pt')

# Para cargar:
modelo_cargado_A = CNN()  # necesitamos instanciar primero
modelo_cargado_A.load_state_dict(torch.load('./pesos_cnn.pt', weights_only=True))
modelo_cargado_A.eval()
print('Opción A (state_dict) — modelo cargado correctamente')

# ── Opción B: Guardar modelo completo ────────────────────────────────────────
# Guarda pesos + arquitectura (serialización de la clase Python).
# Sigue requiriendo la definición de la clase. No es más portable.
# Útil para prototipado rápido.

torch.save(modelo_cnn, './modelo_completo_cnn.pt')

# Para cargar:
modelo_cargado_B = torch.load('./modelo_completo_cnn.pt')
modelo_cargado_B.eval()
print('Opción B (modelo completo) — modelo cargado correctamente')

Opción A (state_dict) — modelo cargado correctamente


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.CNN was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.CNN])` or the `torch.serialization.safe_globals([__main__.CNN])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_o

## 4.2 Pre y post procesado integrados en el modelo

Antes de exportar, encapsulamos todo el procesado dentro del modelo.
En producción, el consumidor solo entrega imágenes crudas y recibe la clase predicha.

In [ ]:
class Preprocesado(torch.nn.Module):
    """Normaliza imágenes crudas (0-255) y añade dimensión de canal."""
    def forward(self, x):
        x = x.float() / 255.0
        x = (x - 0.5) / 0.5
        return x.unsqueeze(1)    # [B, H, W] -> [B, 1, H, W]

class Postprocesado(torch.nn.Module):
    """Convierte logits en probabilidades y clase predicha."""
    def forward(self, x):
        return torch.nn.functional.softmax(x, dim=1), torch.argmax(x, dim=1)

modelo_produccion = torch.nn.Sequential(
    Preprocesado(),
    modelo_cnn.cpu(),
    Postprocesado()
)

# Verificar con imagen cruda
with torch.no_grad():
    x_crudo = torch.randint(0, 256, (4, 48, 48))
    probs, clases = modelo_produccion(x_crudo)
print(f'Probs: {probs.shape} | Clases: {clases}')

## 4.3 TorchScript: `trace` vs `script`

TorchScript genera una representación intermedia ejecutable en C++ sin Python.

- **`trace`**: sigue el flujo de una entrada de ejemplo. Rápido, pero no captura ramas `if/else` ni loops que dependan de los datos.
- **`script`**: analiza el código directamente. Captura cualquier flujo de control. Más robusto.

In [ ]:
# ── Opción 1: trace ───────────────────────────────────────────────────────────
x_ejemplo = torch.randint(0, 256, (1, 48, 48))
traced_model = torch.jit.trace(modelo_produccion, x_ejemplo)
traced_model.save('modelo_traced.zip')
print('trace guardado -> modelo_traced.zip')

# ── Opción 2: script (recomendada si hay control flow) ────────────────────────
scripted_model = torch.jit.script(modelo_produccion)
scripted_model.save('modelo_scripted.zip')
print('script guardado -> modelo_scripted.zip')

# Verificar que el modelo exportado funciona
modelo_ts = torch.jit.load('modelo_scripted.zip')
modelo_ts.eval()
with torch.no_grad():
    probs_ts, clases_ts = modelo_ts(x_ejemplo)
print(f'Verificación TorchScript OK — clase predicha: {clases_ts.item()}')

# Evaluación completa del modelo TorchScript
def evaluar_torchscript(path, dataloader):
    modelo = torch.jit.load(path)
    modelo.eval()
    bar = tqdm(dataloader)
    acc = []
    with torch.no_grad():
        for X, y in bar:
            # desnormalizar para pasar imágenes crudas
            X_crudo = ((X * 0.3081 + 0.1307) * 255).squeeze(1)
            _, clases = modelo(X_crudo)
            acc.append((y == clases).float().mean().item())
            bar.set_description(f'acc {np.mean(acc):.5f}')
    print(f'Precisión TorchScript: {np.mean(acc):.5f}')

evaluar_torchscript('modelo_scripted.zip', dataloaders['test'])

## 4.4 ONNX: máxima portabilidad

ONNX es un formato abierto compatible con TensorFlow, browsers, móviles, IoT.
Permite entrenar en PyTorch y desplegar en prácticamente cualquier entorno.
La desventaja: no todas las operaciones de PyTorch están soportadas.

In [ ]:
torch.onnx.export(
    modelo_produccion,               # modelo con pre/post procesado
    x_ejemplo.float(),               # entrada de ejemplo
    'modelo.onnx',                   # archivo de salida
    export_params=True,              # incluir pesos
    opset_version=11,
    do_constant_folding=True,        # optimización
    input_names=['imagen'],
    output_names=['probabilidades', 'clase'],
    dynamic_axes={
        'imagen'         : {0: 'batch_size'},
        'probabilidades' : {0: 'batch_size'},
        'clase'          : {0: 'batch_size'},
    }
)
print('Modelo exportado -> modelo.onnx')

In [ ]:
# pip install onnxruntime
import onnxruntime

def evaluar_onnx(path, dataloader):
    sesion         = onnxruntime.InferenceSession(path)
    nombre_entrada = sesion.get_inputs()[0].name
    bar            = tqdm(dataloader)
    acc            = []

    with torch.no_grad():
        for X, y in bar:
            # desnormalizar
            X_crudo  = ((X * 0.3081 + 0.1307) * 255).squeeze(1).float().numpy()
            entradas = {nombre_entrada: X_crudo}
            _, clases = sesion.run(None, entradas)
            acc.append((y.numpy() == clases).mean())
            bar.set_description(f'acc {np.mean(acc):.5f}')

    print(f'Precisión ONNX: {np.mean(acc):.5f}')

evaluar_onnx('modelo.onnx', dataloaders['test'])

---
# Resumen

## Redes Neuronales (Notebook 02)
| Concepto | Cuándo usarlo |
|---|---|
| `Sequential` | Arquitecturas lineales simples |
| `Module` custom | Control total sobre el flujo; conexiones residuales |
| Actualización manual de pesos | Para entender la teoría |
| `torch.optim` + `CrossEntropyLoss` | En la práctica siempre |
| Acceso/modificación de capas | Transfer learning |

## Datasets (Notebook 03)
| Nivel | Método | Cuándo |
|---|---|---|
| 1 | Tensor directo | Dataset pequeño, ya en memoria |
| 2 | Mini-batch manual | Entender el concepto |
| 3 | `Dataset` custom | Datos propios, transformaciones |
| 4 | `DataLoader` + `collate_fn` | Siempre en producción |

## Checkpoints y Exportación (Notebook 04)
| Opción | Formato | Cuándo |
|---|---|---|
| `state_dict` | `.pt` | Reanudar entrenamiento, servidor Python |
| Modelo completo | `.pt` | Prototipado rápido |
| TorchScript (trace/script) | `.zip` | Python + C++, sin código extra |
| ONNX | `.onnx` | Máxima portabilidad: web, móvil, IoT |